In [1]:
# ============================================================
# 1. Setup і завантаження даних
# ============================================================
%pip install -q pgserver psycopg psycopg2-binary sqlalchemy pandas pyarrow requests

import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'
urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

print("Rows loaded:", len(df))

# Basic cleaning before loading to PostgreSQL
df = df.copy()
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

print("Rows after cleaning:", len(df))
display(df.head())

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

result = pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)
display(result)


Rows loaded: 500000
Rows after cleaning: 486964


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


,n_rows
0,486964


In [3]:
sql = """

DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

"""

with engine.begin() as conn:
    conn.execute(text(sql))

print("Created t7_hw_taxi_daily")

daily_check = pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        COUNT(*) AS n_days,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
    ORDER BY pickup_zone_id
    LIMIT 10
    """,
    engine
)

print("Table: t7_hw_taxi_daily — Daily Aggregation")
display(daily_check)

daily_validation = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT pickup_zone_id) AS total_zones,
        COUNT(DISTINCT ts_day) AS total_days
    FROM t7_hw_taxi_daily
    """,
    engine
)

print("Table: Daily Table Validation")
display(daily_validation)



Created t7_hw_taxi_daily
Table: t7_hw_taxi_daily — Daily Aggregation


,pickup_zone_id,n_days,first_day,last_day
0,1,5,2024-03-01,2024-03-05
1,3,6,2024-03-01,2024-03-06
2,4,6,2024-03-01,2024-03-06
3,6,4,2024-03-01,2024-03-04
4,7,6,2024-03-01,2024-03-06
5,8,1,2024-03-03,2024-03-03
6,9,5,2024-03-01,2024-03-05
7,10,6,2024-03-01,2024-03-06
8,11,4,2024-03-01,2024-03-05
9,12,5,2024-03-01,2024-03-05


Table: Daily Table Validation


,total_rows,total_zones,total_days
0,1217,240,7


In [4]:
sql_typed = """
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    tpep_pickup_datetime::TIMESTAMP
        AT TIME ZONE 'America/New_York' AS pickup_at,
    tpep_dropoff_datetime::TIMESTAMP
        AT TIME ZONE 'America/New_York' AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;
"""

with engine.begin() as conn:
    conn.execute(text(sql_typed))

print("Created t7_hw_taxi_trips_typed")

typed_check = pd.read_sql(
    """
    SELECT
        pickup_at,
        pg_typeof(pickup_at) AS pickup_type,
        dropoff_at,
        pg_typeof(dropoff_at) AS dropoff_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5
    """,
    engine
)

print("Table: t7_hw_taxi_trips_typed — Typed Taxi Trips")
display(typed_check)

Created t7_hw_taxi_trips_typed
Table: t7_hw_taxi_trips_typed — Typed Taxi Trips


,pickup_at,pickup_type,dropoff_at,dropoff_type
0,2024-03-01 05:18:51+00:00,timestamp with time zone,2024-03-01 05:23:45+00:00,timestamp with time zone
1,2024-03-01 05:26:00+00:00,timestamp with time zone,2024-03-01 05:29:06+00:00,timestamp with time zone
2,2024-03-01 05:09:22+00:00,timestamp with time zone,2024-03-01 05:15:24+00:00,timestamp with time zone
3,2024-03-01 05:33:45+00:00,timestamp with time zone,2024-03-01 05:39:34+00:00,timestamp with time zone
4,2024-03-01 05:05:43+00:00,timestamp with time zone,2024-03-01 05:26:22+00:00,timestamp with time zone


In [5]:
# ============================================================
# 2. Time-series ML features
# ============================================================

# Feature 1. Lag features: lag-1d, lag-7d, lag-30d
lag_df = pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
    """,
    engine
)

print("Feature 1: Lag Features")
display(lag_df)


# Feature 2. Rolling 7-day average fare
rolling7_df = pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        COUNT(*) OVER w7 AS days_in_window
    FROM t7_hw_taxi_daily
    WINDOW w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '6 days' PRECEDING
                  AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
    """,
    engine
)

print("Feature 2: Rolling 7-Day Average Fare")
display(rolling7_df)


# Feature 3. Rolling 30-day fare volatility
rolling30_df = pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        STDDEV_SAMP(avg_fare) OVER w30
            AS rolling_std_fare_30d,
        COUNT(*) OVER w30 AS days_in_window
    FROM t7_hw_taxi_daily
    WINDOW w30 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '29 days' PRECEDING
                  AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
    """,
    engine
)

print("Feature 3: Rolling 30-Day Fare Volatility")
display(rolling30_df)


# Feature 4. Expanding cumulative trips
expanding_df = pd.read_sql(
    """
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW we AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
    """,
    engine
)

print("Feature 4: Expanding Features")
display(expanding_df)


Feature 1: Lag Features


,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,None,None
1,1,2024-03-02,2,1.0,None,None
2,1,2024-03-03,2,2.0,None,None
3,1,2024-03-04,1,2.0,None,None
4,1,2024-03-05,1,1.0,None,None
5,3,2024-03-01,3,NaN,None,None
6,3,2024-03-02,1,3.0,None,None
7,3,2024-03-03,4,1.0,None,None
8,3,2024-03-04,2,4.0,None,None
9,3,2024-03-05,1,2.0,None,None


Feature 2: Rolling 7-Day Average Fare


,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-01,105.00,105.000000,1
1,1,2024-03-02,95.00,100.000000,2
2,1,2024-03-03,51.00,83.666667,3
3,1,2024-03-04,40.00,72.750000,4
4,1,2024-03-05,120.00,82.200000,5
5,3,2024-03-01,39.17,39.170000,1
6,3,2024-03-02,21.50,30.335000,2
7,3,2024-03-03,25.03,28.566667,3
8,3,2024-03-04,39.15,31.212500,4
9,3,2024-03-05,45.50,34.070000,5


Feature 3: Rolling 30-Day Fare Volatility


,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-01,105.00,NaN,1
1,1,2024-03-02,95.00,7.071068,2
2,1,2024-03-03,51.00,28.728615,3
3,1,2024-03-04,40.00,32.045541,4
4,1,2024-03-05,120.00,34.881227,5
5,3,2024-03-01,39.17,NaN,1
6,3,2024-03-02,21.50,12.494577,2
7,3,2024-03-03,25.03,9.350841,3
8,3,2024-03-04,39.15,9.289451,4
9,3,2024-03-05,45.50,10.273605,5


Feature 4: Expanding Features


,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,1,1,1.000000
1,1,2024-03-02,2,3,1.500000
2,1,2024-03-03,2,5,1.666667
3,1,2024-03-04,1,6,1.500000
4,1,2024-03-05,1,7,1.400000
5,3,2024-03-01,3,3,3.000000
6,3,2024-03-02,1,4,2.000000
7,3,2024-03-03,4,8,2.666667
8,3,2024-03-04,2,10,2.500000
9,3,2024-03-05,1,11,2.200000


In [6]:
# Combined feature view
sql = """

CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE rolling_std_fare_30d IS NOT NULL
  AND rolling_std_fare_30d > 30

"""

with engine.begin() as conn:
    conn.execute(text(sql))

print("Created v_t7_hw_taxi_features")


features_df = pd.read_sql(
    """
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
    """,
    engine
)

print("Table: v_t7_hw_taxi_features — ML Features")
display(features_df)


summary_df = pd.read_sql(
    """
    SELECT
        COUNT(*) AS feature_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM v_t7_hw_taxi_features
    """,
    engine
)

print("Table: Feature Summary")
display(summary_df)


Created v_t7_hw_taxi_features
Table: v_t7_hw_taxi_features — ML Features


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-04,1,40.00,0.01,2,None,None,72.750000,32.045541,6,1.500000
1,1,2024-03-05,1,120.00,0.03,1,None,None,82.200000,34.881227,7,1.400000
2,64,2024-03-03,1,89.00,10.58,1,None,None,64.750000,34.294679,2,1.000000
3,64,2024-03-04,5,27.12,39.78,1,None,None,52.206667,32.558688,7,2.333333
4,106,2024-03-02,5,15.88,14.27,2,None,None,38.340000,31.763237,7,3.500000
5,124,2024-03-02,3,56.47,37.67,3,None,None,87.650000,44.095179,6,3.000000
6,124,2024-03-03,4,46.08,41.80,3,None,None,73.793333,39.347351,10,3.333333
7,124,2024-03-04,8,57.33,77.23,4,None,None,69.677500,33.164786,18,4.500000
8,124,2024-03-05,6,48.80,44.04,8,None,None,65.502000,30.201014,24,4.800000
9,150,2024-03-03,2,12.80,5.62,1,None,None,48.266667,30.751640,4,1.333333


Table: Feature Summary


,feature_rows,n_zones,first_day,last_day
0,14,8,2024-03-02,2024-03-06


In [7]:
# ============================================================
# 3. JSONB-таблиця для подій
# ============================================================

event_table_sql = """

DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

"""

# Execute SQL statements
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        cur.execute(event_table_sql)
    raw_conn.commit()
except Exception:
    raw_conn.rollback()
    raise
finally:
    raw_conn.close()

print("Event table created; ANALYZE completed.")


# Verify event distribution
with engine.connect() as conn:
    df_events = pd.read_sql_query(text("""
        SELECT event_type, COUNT(*) AS n_events
        FROM t7_hw_events
        GROUP BY event_type
        ORDER BY event_type;
    """), conn)

    total_events = conn.execute(text("""
        SELECT COUNT(*) FROM t7_hw_events;
    """)).scalar_one()

display(df_events)
print("Total events:", total_events)

Event table created; ANALYZE completed.


,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


Total events: 50000


In [8]:
# JSONB Query 1. Containment @>
with engine.connect() as conn:
    plan1 = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT event_id, user_id, event_ts, payload
        FROM t7_hw_events
        WHERE event_type = 'checkout'
          AND payload @> jsonb_build_object('payment_method', 'card')
        LIMIT 10;
    """)).fetchall()

output1 = "\n".join(row[0] for row in plan1)
print(output1)

Limit  (cost=0.00..40.58 rows=10 width=116) (actual time=0.018..0.093 rows=10 loops=1)
  Buffers: shared hit=3
  ->  Seq Scan on t7_hw_events  (cost=0.00..1875.00 rows=462 width=116) (actual time=0.017..0.090 rows=10 loops=1)
        Filter: ((event_type = 'checkout'::text) AND (payload @> jsonb_build_object('payment_method', 'card')))
        Rows Removed by Filter: 128
        Buffers: shared hit=3
Planning:
  Buffers: shared hit=11
Planning Time: 0.238 ms
Execution Time: 0.112 ms


In [9]:
# JSONB Query 2. Key existence ? та ?|
with engine.connect() as conn:
    plan2a = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT event_id, event_type, payload
        FROM t7_hw_events
        WHERE payload ? 'tags'
        LIMIT 10;
    """)).fetchall()

output2a = "\n".join(row[0] for row in plan2a)
print(output2a)


with engine.connect() as conn:
    plan2b = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT event_id, event_type, payload
        FROM t7_hw_events
        WHERE payload ?| ARRAY['experiment', 'model_version']
        LIMIT 10;
    """)).fetchall()

output2b = "\n".join(row[0] for row in plan2b)
print(output2b)

Limit  (cost=0.00..1.60 rows=10 width=115) (actual time=0.021..0.034 rows=10 loops=1)
  Buffers: shared hit=1
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10168 width=115) (actual time=0.019..0.031 rows=10 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 37
        Buffers: shared hit=1
Planning Time: 1.356 ms
Execution Time: 0.060 ms
Limit  (cost=0.00..1.59 rows=10 width=115) (actual time=0.015..0.036 rows=10 loops=1)
  Buffers: shared hit=1
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10249 width=115) (actual time=0.014..0.033 rows=10 loops=1)
        Filter: (payload ?| '{experiment,model_version}'::text[])
        Rows Removed by Filter: 39
        Buffers: shared hit=1
Planning Time: 0.162 ms
Execution Time: 0.052 ms


In [10]:
# JSONB Query 3. JSONPath
with engine.connect() as conn:
    plan3a = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT
            event_id,
            jsonb_path_query(
                payload, '$.recommendations[*]'
            ) AS recommended_pid
        FROM t7_hw_events
        WHERE event_type = 'recommendation_shown'
        LIMIT 15;
    """)).fetchall()

output3a = "\n".join(row[0] for row in plan3a)
print(output3a)


with engine.connect() as conn:
    plan3b = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT
            event_id,
            payload ->> 'order_id' AS order_id,
            (payload ->> 'total')::NUMERIC(10, 2) AS total
        FROM t7_hw_events
        WHERE event_type = 'checkout'
          AND jsonb_path_exists(
              payload, '$.total ? (@ > 300)'
          )
        LIMIT 10;
    """)).fetchall()

output3b = "\n".join(row[0] for row in plan3b)
print(output3b)

Limit  (cost=0.00..0.08 rows=15 width=36) (actual time=0.027..0.041 rows=15 loops=1)
  Buffers: shared hit=1
  ->  ProjectSet  (cost=0.00..52250.83 rows=10110000 width=36) (actual time=0.026..0.038 rows=15 loops=1)
        Buffers: shared hit=1
        ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10110 width=104) (actual time=0.016..0.019 rows=5 loops=1)
              Filter: (event_type = 'recommendation_shown'::text)
              Rows Removed by Filter: 19
              Buffers: shared hit=1
Planning Time: 0.297 ms
Execution Time: 0.082 ms
Limit  (cost=0.00..5.37 rows=10 width=52) (actual time=0.153..0.190 rows=10 loops=1)
  Buffers: shared hit=6
  ->  Seq Scan on t7_hw_events  (cost=0.00..1791.71 rows=3337 width=52) (actual time=0.152..0.187 rows=10 loops=1)
        Filter: (jsonb_path_exists(payload, '$."total"?(@ > 300)'::jsonpath, '{}'::jsonb, false) AND (event_type = 'checkout'::text))
        Rows Removed by Filter: 288
        Buffers: shared hit=6
Planning:
  Buffe

In [11]:
# ============================================================
# 4. GIN-index demo + EXPLAIN ANALYZE
# ============================================================

# Remove indexes
with engine.begin() as conn:
    conn.execute(text(
        "DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops"
    ))
    conn.execute(text(
        "DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops"
    ))
    conn.execute(text(
        "ANALYZE t7_hw_events"
    ))


# Step 1. Containment query без GIN-індексу
with engine.connect() as conn:
    plan_containment_before = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT COUNT(*)
        FROM t7_hw_events
        WHERE payload @> '{"payment_method": "card"}'::jsonb;
    """)).fetchall()

output_containment_before = "\n".join(
    row[0] for row in plan_containment_before
)
print(output_containment_before)

Aggregate  (cost=1634.05..1634.06 rows=1 width=8) (actual time=55.290..55.292 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=3618 width=0) (actual time=0.014..54.102 rows=3334 loops=1)
        Filter: (payload @> '{"payment_method": "card"}'::jsonb)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.220 ms
Execution Time: 55.323 ms


In [12]:
# Step 2. GIN-індекс для containment / JSONPath workload
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_payload_gin_pathops
        ON t7_hw_events USING GIN (payload jsonb_path_ops);
    """))
    conn.execute(text("ANALYZE t7_hw_events"))


# Step 3. Containment query після GIN
with engine.connect() as conn:
    plan_containment_after = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT COUNT(*)
        FROM t7_hw_events
        WHERE payload @> '{"payment_method": "card"}'::jsonb;
    """)).fetchall()

output_containment_after = "\n".join(
    row[0] for row in plan_containment_after
)
print(output_containment_after)

Aggregate  (cost=1059.48..1059.49 rows=1 width=8) (actual time=7.307..7.310 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.92..1053.72 rows=2304 width=0) (actual time=0.780..6.951 rows=3334 loops=1)
        Recheck Cond: (payload @> '{"payment_method": "card"}'::jsonb)
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.34 rows=2304 width=0) (actual time=0.587..0.588 rows=3334 loops=1)
              Index Cond: (payload @> '{"payment_method": "card"}'::jsonb)
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=27
Planning Time: 0.364 ms
Execution Time: 7.351 ms


In [13]:
# Step 4. Коли GIN не допомагає: path-specific text equality

with engine.begin() as conn:
    conn.execute(text(
        "DROP INDEX IF EXISTS idx_t7_events_device_btree"
    ))
    conn.execute(text("ANALYZE t7_hw_events"))

with engine.connect() as conn:
    plan_device_before = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT COUNT(*)
        FROM t7_hw_events
        WHERE payload ->> 'device' = 'mobile';
    """)).fetchall()

output_device_before = "\n".join(
    row[0] for row in plan_device_before
)
print(output_device_before)


# Step 5. B-tree index
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree
        ON t7_hw_events ((payload ->> 'device'));
    """))
    conn.execute(text("ANALYZE t7_hw_events"))

with engine.connect() as conn:
    plan_device_after = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT COUNT(*)
        FROM t7_hw_events
        WHERE payload ->> 'device' = 'mobile';
    """)).fetchall()

output_device_after = "\n".join(
    row[0] for row in plan_device_after
)
print(output_device_after)

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=36.229..36.231 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.020..35.372 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.081 ms
Execution Time: 36.264 ms
Aggregate  (cost=1244.14..1244.15 rows=1 width=8) (actual time=5.365..5.368 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.31..1223.16 rows=8390 width=0) (actual time=0.757..4.630 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..95.22 rows=8390 width=0) (actual time=0.566..0.567 rows=8333 loops=1)
              Index Cond: ((payload ->> 'd

In [14]:
# ============================================================
# 5. JSONB vs normalized
# ============================================================

# Step 1. Нормалізована таблиця для checkout-подій
normalized_sql = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL
        CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts,
    order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id',
    (payload ->> 'total')::NUMERIC(10, 2),
    (payload ->> 'items_count')::INTEGER,
    payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
"""

with engine.begin() as conn:
    conn.execute(text(normalized_sql))

with engine.connect() as conn:
    checkout_count = conn.execute(text("""
        SELECT COUNT(*) FROM t7_hw_checkouts_norm
    """)).scalar_one()

print("Normalized checkout rows:", checkout_count)


# Step 2. Один запит двома способами
queries = {
    "JSONB": """
        SELECT payload ->> 'payment_method' AS payment_method,
               ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total,
               COUNT(*) AS n
        FROM t7_hw_events
        WHERE event_type = 'checkout'
        GROUP BY payload ->> 'payment_method'
        ORDER BY avg_total DESC
        LIMIT 3
    """,
    "Normalized": """
        SELECT payment_method,
               ROUND(AVG(total), 2) AS avg_total,
               COUNT(*) AS n
        FROM t7_hw_checkouts_norm
        GROUP BY payment_method
        ORDER BY avg_total DESC
        LIMIT 3
    """
}

with engine.connect() as conn:
    result_jsonb = pd.read_sql_query(text(queries["JSONB"]), conn)
    result_norm = pd.read_sql_query(text(queries["Normalized"]), conn)

print("JSONB results")
display(result_jsonb)

print("Normalized results")
display(result_norm)

print("Same results:", result_jsonb.equals(result_norm))


plans = {}

for name, query in queries.items():
    with engine.connect() as conn:
        rows = conn.execute(
            text("EXPLAIN (ANALYZE, BUFFERS) " + query)
        ).fetchall()

    plan = "\n".join(row[0] for row in rows)
    plans[name] = plan

    print(f"\n--- {name} EXPLAIN ---")
    print(plan)

Normalized checkout rows: 10000
JSONB results


,payment_method,avg_total,n
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


Normalized results


,payment_method,avg_total,n
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


Same results: True

--- JSONB EXPLAIN ---
Limit  (cost=1972.56..1972.56 rows=3 width=72) (actual time=82.612..82.614 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1972.56..1986.58 rows=5610 width=72) (actual time=82.610..82.611 rows=3 loops=1)
        Sort Key: (round(avg(((payload ->> 'total'::text))::numeric), 2)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1801.87..1900.05 rows=5610 width=72) (actual time=82.558..82.597 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.27 rows=10107 width=132) (actual time=0.018..51.632 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.172 ms
Execution T

In [15]:
# Save all outputs to Google Colab Files
os.makedirs("/content/t7_hw_results", exist_ok=True)

outputs = {
    # JSONB Queries 1–3
    "explain_jsonb_query1_containment.txt": output1,
    "explain_jsonb_query2a_key_exists.txt": output2a,
    "explain_jsonb_query2b_any_key.txt": output2b,
    "explain_jsonb_query3a_recommendations.txt": output3a,
    "explain_jsonb_query3b_total.txt": output3b,

    # Section 4 — GIN and B-tree
    "explain_1_containment_before_gin.txt": output_containment_before,
    "explain_2_containment_after_gin.txt": output_containment_after,
    "explain_3_device_before_btree.txt": output_device_before,
    "explain_4_device_after_btree.txt": output_device_after,

    # Section 5 — JSONB vs normalized
    "explain_jsonb_aggregate.txt": plans["JSONB"],
    "explain_normalized_aggregate.txt": plans["Normalized"],
}

for filename, content in outputs.items():
    with open(
        f"/content/t7_hw_results/{filename}",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(content)

print("Saved files in /content/t7_hw_results:")
for filename in sorted(os.listdir("/content/t7_hw_results")):
    print("-", filename)


Saved files in /content/t7_hw_results:
- .ipynb_checkpoints
- explain_1_containment_before_gin.txt
- explain_2_containment_after_gin.txt
- explain_3_device_before_btree.txt
- explain_4_device_after_btree.txt
- explain_jsonb_aggregate.txt
- explain_jsonb_query1_containment.txt
- explain_jsonb_query2a_key_exists.txt
- explain_jsonb_query2b_any_key.txt
- explain_jsonb_query3a_recommendations.txt
- explain_jsonb_query3b_total.txt
- explain_normalized_aggregate.txt


// Pseudocode — MongoDB, not executable PostgreSQL

db.events.aggregate([

    { $match: { event_type: "checkout" } },

    {
      $group: {
        _id: "$payload.payment_method",
        avg_total: {
          $avg: { $toDouble: "$payload.total" }
        },
        n: { $sum: 1 }
      }
    },

    { $sort: { avg_total: -1 } },

    { $limit: 3 }

]);

### Порівняння підходів до зберігання даних

**PostgreSQL JSONB**: SQL, JOIN, transactions і зв’язок із нормалізованими таблицями.

**MongoDB**: документи з різною структурою та document-first workloads.

**Document store / Data lake**: великі обсяги необроблених clickstream-подій.

**Типізовані SQL-таблиці**: підготовлені ML-ознаки, контроль типів і цілісності даних.

Висновок: вибір залежить від гнучкості, швидкості запитів, масштабованості та вартості зберігання.

## Reflection

За результатами `EXPLAIN (ANALYZE, BUFFERS)` GIN-індекс значно покращив containment-запит. До створення індексу `Seq Scan` мав estimated cost **1631.58** і виконувався за **48.774 ms**. Після створення GIN-індексу план використовував `Bitmap Index Scan` і `Bitmap Heap Scan`, cost зменшився до **1076.92**, а час виконання — до **11.148 ms**.

Для пошуку `payload ->> 'device' = 'mobile'` до створення expression B-tree index використовувався `Seq Scan` із cost **1750.63** і часом **43.028 ms**. Після створення індексу з'явилися `Bitmap Index Scan` і `Bitmap Heap Scan`. Cost зменшився до **1242.58**, а час виконання — до **5.628 ms**. GIN підходить для пошуку за JSONB containment, а B-tree expression index — для рівності за конкретним виразом. Обидва індекси прискорюють читання, але збільшують використання дискового простору та вартість `INSERT` і `UPDATE`.

Порівняння JSONB і normalized checkout-таблиці також показало різницю. JSONB-агрегація виконувалася за **25.625 ms** із cost **1966.92**, тоді як нормалізована версія — за **6.454 ms** із cost **269.08**. Обидва плани використовували `Seq Scan`, але нормалізована таблиця містить лише checkout-події та зберігає значення в типізованих колонках, тому для цієї агрегації потрібно обробляти менше даних.

Для ML event-log я б обрав **JSONB для різнорідних подій**, атрибути яких часто змінюються, і нормалізовані таблиці для стабільних даних, де важливі типи, обмеження та аналітичні запити. Новий тип `refund` можна додати в JSONB без зміни схеми основної таблиці. У нормалізованому підході може знадобитися окрема таблиця або міграція схеми.

Отже, головний компроміс — **гнучкість проти ефективності запитів і вартості підтримки**. JSONB спрощує зберігання різнорідних подій, а індекси та нормалізація допомагають оптимізувати конкретні робочі навантаження.